# 📖 LLM Limitations and Common Errors

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** hallucinations and why they happen
2. **Identify** context window limitations
3. **Handle** non-determinism in production
4. **Deal with** rate limits and API errors

---

## ⏱️ Time Estimate

**~25 minutes**

---

## 📦 Setup

In [ ]:
!pip install -q openai
import os
if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

from openai import OpenAI
client = OpenAI()

---

## 👻 Hallucinations: When LLMs Make Things Up

A **hallucination** is when the LLM generates false or nonsensical information that appears true.

### Why Do Hallucinations Happen?

```
┌─────────────────────────────────────────────────────┐
│              WHY HALLUCINATIONS HAPPEN               │
├─────────────────────────────────────────────────────┤
│  1. LLM predicts next token, not "truth"            │
│  2. Training data may contain misinformation         │
│  3. Model doesn't know what it doesn't know        │
│  4. No real understanding or memory of facts       │
│  5. Confidence doesn't match accuracy              │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# Example: Classic hallucination - fake citations
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What does research show about AI in healthcare? Cite 3 sources."}],
    max_tokens=200
)
print("👻 HALLUCINATION EXAMPLE - Fake Citations")
print("=" * 60)
print(response.choices[0].message.content)

### How to Reduce Hallucinations

| Technique | Description | When to Use |
|-----------|-------------|-------------|
**| Provide context | Give the model sources to work with | Factual tasks |
**| Use Chain of Thought | Ask for step-by-step reasoning | Complex reasoning |
**| Lower temperature | Set to 0 for factual tasks | Code, math |
**| Validate outputs | Cross-check with external sources | Critical applications |
**| Prompt for uncertainty | Tell it to say "I don't know" | When unsure matters |

In [ ]:
# Reducing hallucinations with better prompts
system_prompt = """You are a helpful assistant. 
IMPORTANT: If you're not sure about something, say "I don't know" or "I'm not certain."
Never make up facts, statistics, or citations. 
If asked for sources, only mention real, verified sources you know."""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What does research show about AI in healthcare? Cite sources."}
    ],
    max_tokens=200
)
print("✅ REDUCED HALLUCINATIONS - Asking for uncertainty")
print("=" * 60)
print(response.choices[0].message.content)

---

## 📊 Context Window Limitations

Every LLM has a maximum context window - the total amount of text it can consider:

In [ ]:
# What happens when context is too long?
long_context = "x " * 15000  # Simulating very long text

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": f"What is the last word in this text? {long_context}"}
    ],
    max_tokens=50
)
print("⚠️ CONTEXT WINDOW LIMIT")
print("=" * 60)
print(f"Response: {response.choices[0].message.content}")
print("\nNote: Long inputs may be truncated or cause errors!")

### Context Window Strategies

```
┌─────────────────────────────────────────────────────┐
│             HANDLING LONG CONTEXT                   │
├─────────────────���─��─────────────────────────────────┤
│  1. Summarize older messages                       │
│  2. Use sliding window approach                   │
│  3. Feed most important info at start AND end      │
│  4. Chunk long documents into sections             │
│  5. Use RAG (Retrieval Augmented Generation)      │
└─────────────────────────────────────────────────────┘
```

---

## 🎲 Non-Determinism: Why Same Input = Different Outputs

LLMs are probabilistic - the SAME prompt can produce DIFFERENT results:

In [ ]:
# Run the same prompt multiple times with temperature > 0
prompt = "Tell me a short, unique nickname for Python."

print("🎲 NON-DETERMINISM DEMO")
print("=" * 60)
print(f"Prompt: '{prompt}'\n")

responses = []
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0.7  # This creates variation!
    )
    result = response.choices[0].message.content
    responses.append(result)
    print(f"Run {i+1}: {result}")

print(f"\n⚠️ All different! Same input, different outputs.")

### Handling Non-Determinism in Production

| Strategy | When to Use |
|---------|--------------|
**| temperature=0 | Code generation, math, exact answers |
**| seed parameter | Reproducible results (if supported) |
**| max_tokens limit | Prevent runaway outputs |
**| Output validation | Check outputs meet criteria |
**| Retry logic | Handle unexpected outputs |

**Key**: For agent evals, understanding non-determinism is CRITICAL - we'll cover this in detail!

---

## 🌊 Rate Limits and API Errors

Production systems must handle API errors gracefully:

In [ ]:
import time
from openai import RateLimitError, APIError

# Example: Handling rate limits with retry logic
def call_with_retry(client, prompt, max_retries=3):
    """Call OpenAI with exponential backoff retry."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            wait_time = 2 ** attempt  # Exponential backoff
            print(f"⚠️ Rate limit hit. Waiting {wait_time}s...")
            time.sleep(wait_time)
        except APIError as e:
            print(f"❌ API Error: {e}")
            raise
    return "Max retries exceeded"

# Quick test
result = call_with_retry(client, "Hello!")
print(f"✅ Result: {result}")

### Common API Errors

| Error | Cause | Solution |
|-------|-------|----------|
**| RateLimitError | Too many requests | Add retry with backoff |
**| InvalidRequestError | Bad parameters | Check model/messages format |
**| AuthenticationError | Bad API key | Verify key is correct |
**| APIConnectionError | Network issues | Retry logic |

---

## 🐛 Debugging LLM Responses

Let's see how to debug and understand what's happening:

In [ ]:
import json

# Get full response for debugging
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)

# Inspect the full object
print("🔍 DEBUGGING: Full Response Object")
print("=" * 60)
print(json.dumps({
    "model": response.model,
    "created": response.created,
    "finish_reason": response.choices[0].finish_reason,
    "usage": {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens
    }
}, indent=2))

---

## ⚡ Output Truncation

If max_tokens is too low, output gets cut off:

In [ ]:
# Compare different max_tokens
prompt = "Explain Python async/await in detail."

for max_tok in [20, 50, 100]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok
    )
    finish = response.choices[0].finish_reason
    print(f"max_tokens={max_tok:3d} → finish_reason: {finish}")
    print(f"   Output: {response.choices[0].message.content[:50]}...")
    print()

### Stop Reasons

| Reason | Meaning |
|--------|----------|
**| stop | Model chose to stop naturally |
**| length | Hit max_tokens limit |
**| content_filter | Content was filtered |

---

## 🧪 Try It Yourself!

**Exercise 1**: Prompt the model to give you false information, then try to reduce it.
**Exercise 2**: Set temperature to 0 and run the same prompt 5 times - verify same output.
**Exercise 3**: Create a system that tracks rate limits and backs off.

In [ ]:
# 🧪 Exercise: Try to make the model hallucinate, then prevent it

# YOUR CODE HERE

system_hallucination = "You are a research assistant who always cites real research papers. If you don't know a paper, say so."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_hallucination},
        {"role": "user", "content": "Tell me about a research paper on quantum computing in 2024."}
    ],
    max_tokens=200
)
print("Result:", response.choices[0].message.content)

---

## ❓ FAQ

**Q: Can hallucinations be completely eliminated?**
A: No, but they can be significantly reduced with proper prompting and RAG.

**Q: Why does my code sometimes fail?**
A: Probabilistic output - use temperature=0 for reproducible code.

**Q: What's the best way to handle long documents?**
A: Use RAG (Retrieval Augmented Generation) - we'll cover this later!

**Q: How do I debug unexpected outputs?**
A: Check usage data, finish_reason, and try with lower temperature.

---

## ✅ Summary

You now understand:

1. **Hallucinations** - Model makes things up; reduce with context + uncertainty prompts
2. **Context window** - Max tokens limit; use summarization or RAG
3. **Non-determinism** - Same input can give different outputs; use temperature=0 for consistency
4. **API errors** - Rate limits, bad requests; implement retry logic
5. **Output truncation** - May get cut off; adjust max_tokens

**This knowledge is CRITICAL for agent evals** - understanding what can go wrong is the first step to evaluating it!

---

## 🔗 Next Steps

Next notebook: **[01_what_are_agents.ipynb](../PART_2_Agents_Basics/01_what_are_agents.ipynb)** - Learn what agents are and why they're different from raw LLMs!

---

*Congratulations! You've completed PART 1: LLM Fundamentals! 🎉*